# 01 — TF-IDF Baseline Experiments

This notebook validates and tunes the TF-IDF component used in `app/ai/tfidf_model.py`.

**What we explore:**
- How n-gram range affects keyword matching quality
- Effect of `sublinear_tf` (log normalization) on scores
- Top keyword extraction from sample job descriptions
- Raw TF-IDF cosine similarity scores before scaling

**Production config (in `tfidf_model.py`):**
```python
TfidfVectorizer(ngram_range=(1,2), max_features=10_000, sublinear_tf=True)
Final score = min(1.0, raw_tfidf * 3.0)   # scaling applied in hybrid_scorer.py
```

In [ ]:
import sys
sys.path.append('..')   # so we can import from app/

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from app.ai.preprocess import clean_text
from app.ai.tfidf_model import compute_tfidf_score, get_top_keywords

print('Imports OK')

## 1. Sample Data

In [ ]:
JOB_DESC = """
We are looking for a Backend Software Engineer with strong experience in Python, Flask, and REST API design.
You should have hands-on experience with MongoDB, Docker, and cloud deployments (AWS or GCP).
Experience with JWT authentication, WebSocket communication, and CI/CD pipelines is required.
Familiarity with NLP libraries such as scikit-learn, spaCy, or sentence-transformers is a plus.
"""

CV_HIGH_MATCH = """
John Smith — Backend Engineer
5 years experience with Python and Flask REST APIs. Built and deployed MongoDB-backed microservices on AWS.
Implemented JWT authentication and WebSocket integrations. Containerized apps using Docker.
Used scikit-learn for classification tasks and spaCy for NER.
"""

CV_PARTIAL_MATCH = """
Jane Doe — Full Stack Developer
3 years of experience with Node.js and Express. Comfortable with SQL and PostgreSQL databases.
Used React for frontend applications. Basic knowledge of REST APIs. Familiar with Git and CI/CD.
"""

CV_LOW_MATCH = """
Tom Brown — Graphic Designer
Expert in Adobe Photoshop, Illustrator, and InDesign. Created visual content for social media campaigns.
Experienced in brand identity, typography, and motion graphics.
"""

samples = {
    'High Match CV': CV_HIGH_MATCH,
    'Partial Match CV': CV_PARTIAL_MATCH,
    'Low Match CV': CV_LOW_MATCH
}

print('Sample data ready.')

## 2. N-gram Range Comparison

Comparing `(1,1)` unigrams only vs `(1,2)` unigrams+bigrams (production config) vs `(1,3)`.

In [ ]:
ngram_ranges = [(1,1), (1,2), (1,3)]
results = {}

for label, cv_text in samples.items():
    row = []
    cleaned_jd = clean_text(JOB_DESC)
    cleaned_cv = clean_text(cv_text)
    for ng in ngram_ranges:
        vec = TfidfVectorizer(ngram_range=ng, max_features=10_000, sublinear_tf=True)
        mat = vec.fit_transform([cleaned_jd, cleaned_cv])
        raw_score = cosine_similarity(mat[0:1], mat[1:2])[0][0]
        scaled_score = min(1.0, raw_score * 3.0) * 100
        row.append(round(scaled_score, 2))
    results[label] = row

df = pd.DataFrame(results, index=[f'ngram {n}' for n in ngram_ranges]).T
print(df.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(samples))
width = 0.25
colors = ['#4f86c6', '#f4845f', '#67b99a']

for i, ng in enumerate(ngram_ranges):
    vals = [results[label][i] for label in samples]
    ax.bar(x + i*width, vals, width, label=f'ngram {ng}', color=colors[i])

ax.set_xlabel('Resume Type')
ax.set_ylabel('TF-IDF Score (scaled, %)')
ax.set_title('TF-IDF Score by N-gram Range')
ax.set_xticks(x + width)
ax.set_xticklabels(samples.keys())
ax.legend()
ax.set_ylim(0, 110)
plt.tight_layout()
plt.show()

print('Production uses (1,2) — bigrams capture phrases like "REST API", "machine learning", "Docker" better than unigrams.')

## 3. Sublinear TF vs Raw TF

In [ ]:
cleaned_jd = clean_text(JOB_DESC)

comparison = []
for label, cv_text in samples.items():
    cleaned_cv = clean_text(cv_text)
    for use_sublinear in [False, True]:
        vec = TfidfVectorizer(ngram_range=(1,2), max_features=10_000, sublinear_tf=use_sublinear)
        mat = vec.fit_transform([cleaned_jd, cleaned_cv])
        raw = cosine_similarity(mat[0:1], mat[1:2])[0][0]
        comparison.append({
            'CV': label,
            'sublinear_tf': use_sublinear,
            'raw_score': round(raw, 4),
            'scaled_%': round(min(1.0, raw * 3.0) * 100, 2)
        })

df_cmp = pd.DataFrame(comparison)
print(df_cmp.to_string(index=False))

## 4. Top Keywords Extracted from Job Description

In [ ]:
top_kw = get_top_keywords(JOB_DESC, top_n=20)
print('Top 20 TF-IDF keywords from JD:')
for i, kw in enumerate(top_kw, 1):
    print(f'  {i:2d}. {kw}')

## 5. Score Distribution on Real Dataset

In [ ]:
import os

dataset_path = os.path.join('..', 'data', 'UpdatedResumeDataSet.csv')
if os.path.exists(dataset_path):
    df_real = pd.read_csv(dataset_path)
    print(f'Loaded {len(df_real)} resumes. Columns: {list(df_real.columns)}')
    print(df_real.head(3))
else:
    print('Dataset not found at expected path.')

In [ ]:
# Score a sample of 50 resumes against the same JD and plot the distribution
if 'df_real' in dir() and 'Resume' in df_real.columns:
    sample = df_real.sample(min(50, len(df_real)), random_state=42)
    scores = [compute_tfidf_score(JOB_DESC, str(r)) * 100 for r in sample['Resume']]

    plt.figure(figsize=(9, 4))
    plt.hist(scores, bins=20, color='#4f86c6', edgecolor='white')
    plt.axvline(np.mean(scores), color='red', linestyle='--', label=f'Mean = {np.mean(scores):.1f}%')
    plt.title('TF-IDF Score Distribution (50-resume sample)')
    plt.xlabel('TF-IDF Score (%)')
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Skipping — dataset not available.')

## Summary

| Config | Observation |
|--------|-------------|
| `ngram_range=(1,2)` | Best discrimination — bigrams catch multi-word skills |
| `sublinear_tf=True` | Reduces dominance of high-frequency terms, fairer scoring |
| Scaling `× 3.0` | Raw cosine values cluster near 0.1–0.3; ×3 spreads them to 0–1 range |

**Conclusion:** Production config `(1,2)` + `sublinear_tf=True` + `×3 scaling` gives the best balance of precision and range.